In [2]:
import geopandas as gpd
from pathlib import Path

# Carga el shapefile de pluviómetros SAIH (descargado manualmente del MITECO)
ruta_shp = Path("../data/raw/saih_inventario/capassaih-pluviometria/Pluviometria.shp")
pluv = gpd.read_file(ruta_shp)

print(f"Total pluviómetros SAIH en España: {len(pluv)}")
print(f"Sistema de coordenadas: {pluv.crs}")
print(f"\nColumnas: {list(pluv.columns)}")
pluv.head()

Total pluviómetros SAIH en España: 1353
Sistema de coordenadas: EPSG:4258

Columnas: ['OBJECTID', 'IDPUNTO', 'DESCRIPCIO', 'IDSAIH', 'TIPO', 'geometry']


,OBJECTID,IDPUNTO,DESCRIPCIO,IDSAIH,TIPO,geometry
0,963,I07_SALADO_MENG,I07_SALADO_MENG,GUADALQUIVIR,Pluviometría (mm),POINT (-3.8075 37.98144)
1,978,AR49 AZUD DEL ENCIN,AR49 AZUD DEL ENCIN,TAJO,Pluviometría (mm),POINT (-6.42594 39.94885)
2,994,AR22 COFIO EN SAN MARTIN,AR22 COFIO EN SAN MARTIN,TAJO,Pluviometría (mm),POINT (-4.32259 40.41204)
3,1011,E_11 BELENA,E_11 BELENA,TAJO,Pluviometría (mm),POINT (-3.19433 40.93034)
4,1033,EM3-02,CUMBRE MAYORES,GUADIANA,Pluviometría (mm),POINT (-6.64619 38.05858)


In [3]:
# Reproyectar a WGS84 (lat/lon) si está en otra proyección
if pluv.crs and pluv.crs.to_epsg() != 4326:
    pluv = pluv.to_crs(epsg=4326)

# Inspecciona qué valores únicos hay en columnas categóricas
for col in pluv.columns:
    if col != "geometry" and pluv[col].dtype == "object":
        unicos = pluv[col].nunique()
        if unicos < 20:  # solo columnas con pocas categorías (posibles filtros)
            print(f"\n{col} ({unicos} únicos):")
            print(pluv[col].value_counts())

In [4]:
print(pluv["IDSAIH"].unique())
print(f"\nTotal demarcaciones: {pluv['IDSAIH'].nunique()}")
print(f"\nPluviómetros por demarcación:")
print(pluv["IDSAIH"].value_counts())

<ArrowStringArray>
[       'GUADALQUIVIR',                'TAJO',            'GUADIANA',
                'EBRO',                 'ACA',              'NORTEI',
               'JUCAR',              'SEGURA', 'AGENCIAANDALUZAAGUA',
               'DUERO']
Length: 10, dtype: str

Total demarcaciones: 10

Pluviómetros por demarcación:
IDSAIH
EBRO                   356
GUADALQUIVIR           198
TAJO                   191
GUADIANA               186
JUCAR                  124
AGENCIAANDALUZAAGUA    103
NORTEI                  81
SEGURA                  65
ACA                     48
DUERO                    1
Name: count, dtype: int64


In [5]:
# Filtra al Júcar (ajusta el valor si la celda anterior te dice otra escritura)
saih_jucar = pluv[pluv["IDSAIH"] == "JUCAR"].copy()

# Extrae lat y lon de la geometría
saih_jucar["lon"] = saih_jucar.geometry.x
saih_jucar["lat"] = saih_jucar.geometry.y

print(f"Pluviómetros SAIH-CHJ: {len(saih_jucar)}")
saih_jucar[["IDPUNTO", "DESCRIPCIO", "lat", "lon"]].head(10)

Pluviómetros SAIH-CHJ: 124


,IDPUNTO,DESCRIPCIO,lat,lon
53,PLUVIO VALLIBONA,VALLIBONA,40.584792,0.001111
54,PLUVIO VALBONA,VALBONA,40.254068,-0.818062
55,PLUVIO PINA,PINA,40.029234,-0.628123
56,PLUVIO ALBARRACIN,ALBARRACIN,40.431020,-1.436050
57,PLUVIO LANDETE,LANDETE,39.911618,-1.371545
58,PLUVIO BENAGEBER,BENAGEBER,39.727324,-1.099924
59,PLUVIO ZAGRA,ZAGRA,39.843047,-1.144831
86,PLUVIO LA CUEVARRUZ,LA CUEVARRUZ,39.936402,-0.984184
87,PLUVIO FORATA,FORATA,39.341183,-0.864723
88,PLUVIO COFRENTES,COFRENTES,39.263031,-1.096256


In [6]:
import json
import pandas as pd
import sys
sys.path.insert(0, "..")
from src.ingest.aemet import parse_coord

with open("../data/raw/aemet/inventario_estaciones.json", encoding="utf-8") as f:
    inv = pd.DataFrame(json.load(f))

provincias_jucar = ["VALENCIA", "CUENCA", "ALBACETE", "CASTELLON", "TERUEL"]
aemet_jucar = inv[inv["provincia"].isin(provincias_jucar)].copy()
aemet_jucar["lat"] = aemet_jucar["latitud"].apply(parse_coord)
aemet_jucar["lon"] = aemet_jucar["longitud"].apply(parse_coord)

print(f"Estaciones AEMET en la zona: {len(aemet_jucar)}")
print(f"Pluviómetros SAIH-CHJ: {len(saih_jucar)}")
print(f"Total combinado: {len(aemet_jucar) + len(saih_jucar)} puntos")

Estaciones AEMET en la zona: 77
Pluviómetros SAIH-CHJ: 124
Total combinado: 201 puntos


In [7]:
import folium
from pathlib import Path

# Estaciones AEMET que descargaste durante el TFM
descargadas_aemet = {"8414A", "8293X", "8175", "8416Y", "8500A", "8025", "8203O", "8058X"}

m = folium.Map(location=[39.5, -1.2], zoom_start=8, tiles="CartoDB positron")

# --- AEMET en rojo ---
for _, row in aemet_jucar.iterrows():
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue
    es_descargada = row["indicativo"] in descargadas_aemet
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8 if es_descargada else 4,
        color="darkred",
        fill=True,
        fill_color="red",
        fill_opacity=0.8 if es_descargada else 0.5,
        weight=2 if es_descargada else 1,
        popup=f"<b>AEMET — {row['nombre']}</b><br>Código: {row['indicativo']}<br>Provincia: {row['provincia']}",
    ).add_to(m)

# --- SAIH-CHJ en azul ---
for _, row in saih_jucar.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color="navy",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        weight=1,
        popup=f"<b>SAIH — {row['DESCRIPCIO']}</b><br>ID: {row['IDPUNTO']}",
    ).add_to(m)

# Leyenda fija en la esquina inferior-izquierda
legend_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    width: 220px;
    background: white;
    border: 2px solid #555;
    border-radius: 6px;
    padding: 12px;
    font-size: 13px;
    font-family: Arial, sans-serif;
    z-index: 9999;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    line-height: 1.6;
">
<b style="font-size:14px;">Red de observación</b><br>
<span style="display:inline-block; width:14px; text-align:center;">
  <svg width="14" height="14" style="vertical-align:middle">
    <circle cx="7" cy="7" r="6" fill="red" fill-opacity="0.85"
            stroke="darkred" stroke-width="1.5"/>
  </svg>
</span>
AEMET (descargada)<br>
<span style="display:inline-block; width:14px; text-align:center;">
  <svg width="14" height="14" style="vertical-align:middle">
    <circle cx="7" cy="7" r="4" fill="red" fill-opacity="0.5"
            stroke="darkred" stroke-width="1"/>
  </svg>
</span>
AEMET (disponible)<br>
<span style="display:inline-block; width:14px; text-align:center;">
  <svg width="14" height="14" style="vertical-align:middle">
    <circle cx="7" cy="7" r="3" fill="blue" fill-opacity="0.7"
            stroke="navy" stroke-width="1"/>
  </svg>
</span>
SAIH-CHJ
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# Guarda HTML
out_path = Path("../data/processed/mapa_aemet_saih.html")
out_path.parent.mkdir(parents=True, exist_ok=True)
m.save(str(out_path))
print(f"Mapa guardado en: {out_path.resolve()}")

m

Mapa guardado en: C:\Users\Maria\Documents\tfm-flood-warning\data\processed\mapa_aemet_saih.html


In [8]:
# Veamos cómo se llaman las estaciones del Júcar en el shapefile
print("=== Muestra de DESCRIPCIO (Júcar) ===")
print(saih_jucar["DESCRIPCIO"].head(30).to_list())
print()
print("=== Muestra de IDPUNTO (Júcar) ===")
print(saih_jucar["IDPUNTO"].head(30).to_list())

=== Muestra de DESCRIPCIO (Júcar) ===
['VALLIBONA', 'VALBONA', 'PINA', 'ALBARRACIN', 'LANDETE', 'BENAGEBER', 'ZAGRA', 'LA CUEVARRUZ', 'FORATA', 'COFRENTES', 'ESTUBENY', 'AYORA', 'ONTINYENT', 'CASA DEL BARON', 'MARINA BAIXA', 'ABDET', 'BENIARRES', 'VERNISSA', 'EL PUIG', 'ALFONDEGUILLA', 'GUDAR', 'CASTELFRIO', 'ZARZUELA', 'PAJARONCILLO', 'CUERDA', 'REMEDIO CHELVA', 'ANTELLA', 'MOIXENT', 'VILLAMALEA', 'AYORA LA HUNDE']

=== Muestra de IDPUNTO (Júcar) ===
['PLUVIO  VALLIBONA', 'PLUVIO  VALBONA', 'PLUVIO  PINA', 'PLUVIO  ALBARRACIN', 'PLUVIO  LANDETE', 'PLUVIO  BENAGEBER', 'PLUVIO  ZAGRA', 'PLUVIO  LA CUEVARRUZ', 'PLUVIO  FORATA', 'PLUVIO  COFRENTES', 'PLUVIO  ESTUBENY', 'PLUVIO  AYORA', 'PLUVIO  ONTINYENT', 'PLUVIO  CASA DEL BARON', 'PLUVIO  MARINA BAIXA', 'PLUVIO  ABDET', 'PLUVIO  BENIARRES', 'PLUVIO  VERNISSA', 'PLUVIO  EL PUIG', 'PLUVIO  ALFONDEGUILLA', 'PLUVIO  GUDAR', 'PLUVIO  CASTELFRIO', 'PLUVIO  ZARZUELA', 'PLUVIO  PAJARONCILLO', 'PLUVIO  CUERDA', 'PLUVIO  REMEDIO CHELVA', 'PLUVIO 

In [12]:
import unicodedata
import pandas as pd


def normalizar(s):
    """Mayúsculas, sin tildes, espacios simples. Para comparar nombres."""
    if not isinstance(s, str):
        return ""
    s = s.upper().strip()
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return " ".join(s.split())


# Cargar los datos del PDF (Parquet histórico)
saih_pdf = pd.read_parquet("../data/processed/saih_historico_2020_2024.parquet")

# Normalizar nombres del shapefile (ya tienes saih_jucar cargado)
saih_jucar["nombre_norm"] = saih_jucar["DESCRIPCIO"].apply(normalizar)

# Lista de nombres del shapefile, ordenados de más largo a más corto.
# Importante: primero los más largos para que "AYORA LA HUNDE" gane sobre "AYORA"
# cuando la descripción del PDF dice "Pluviómetro de Ayora La Hunde Ayora".
nombres_shp = sorted(
    [n for n in saih_jucar["nombre_norm"].dropna().unique() if n],
    key=len,
    reverse=True,
)


def buscar_match(descripcion_pdf):
    """Devuelve el nombre normalizado del shapefile que aparece en la descripción."""
    desc_norm = normalizar(descripcion_pdf)
    for n in nombres_shp:
        if n in desc_norm:
            return n
    return None


# Aplicar a todas las descripciones únicas de los PDFs
descripciones = (saih_pdf["descripcion"]
                    .drop_duplicates()
                    .to_frame()
                    .reset_index(drop=True))
descripciones["match"] = descripciones["descripcion"].apply(buscar_match)

total = len(descripciones)
con_match = descripciones["match"].notna().sum()

print(f"Descripciones únicas en los PDFs: {total}")
print(f"  Con match al shapefile: {con_match} ({con_match/total*100:.1f}%)")
print(f"  Sin match: {total - con_match} ({(total-con_match)/total*100:.1f}%)")
print()
print("=== Ejemplos de matches OK ===")
print(descripciones[descripciones["match"].notna()].head(15).to_string())
print()
print("=== Ejemplos SIN match (a revisar) ===")
print(descripciones[descripciones["match"].isna()].head(15).to_string())

Descripciones únicas en los PDFs: 186
  Con match al shapefile: 113 (60.8%)
  Sin match: 73 (39.2%)

=== Ejemplos de matches OK ===
                                               descripcion         match
1               Embalse de Ulldecona La Pobla de Benifassa     ULLDECONA
2          Pluvionivómetro de Fredes La Pobla de Benifassa        FREDES
3                     Pluvionivómetro de Vallibona Morella     VALLIBONA
5                                 Pluviómetro de Catí Catí          CATI
6                                 Pluviómetro de Xert Xert          XERT
9                                   Aforo en Cuenca Cuenca        CUENCA
12                      Aforo en Pajaroncillo Pajaroncillo  PAJARONCILLO
14                              Embalse de Alarcón Alarcón       ALARCON
15                                  Embalse de la Toba Uña       LA TOBA
18            Pluvionivometro Tío Calores Barchin del Hoyo   TIO CALORES
19            Pluvionivometro de Talayuelo Arcas de Villar     TA

he tenido que descargar la info de todo porque en saih viene de caudales rios y pluviomentros. solo teniamos pluviomntros.

In [11]:
import geopandas as gpd
import pandas as pd

# Cargar las 3 capas (ajusta los nombres exactos de carpeta/fichero según lo que tengas)
base = "../data/raw/saih_inventario"
pluv = gpd.read_file(f"{base}/capassaih-pluviometria/Pluviometria.shp")
emba = gpd.read_file(f"{base}/capassaih-embalses/Embalses.shp")
caud = gpd.read_file(f"{base}/capassaih-rios/Caudales.shp")

# Etiqueta el tipo en cada uno antes de combinar
pluv["TIPO_CAPA"] = "PLUVIOMETRO"
emba["TIPO_CAPA"] = "EMBALSE"
caud["TIPO_CAPA"] = "AFORO"

# Combinar todo
saih_all = gpd.GeoDataFrame(pd.concat([pluv, emba, caud], ignore_index=True))

# Filtrar al Júcar
saih_jucar = saih_all[saih_all["IDSAIH"] == "JUCAR"].copy()
saih_jucar["lon"] = saih_jucar.geometry.x
saih_jucar["lat"] = saih_jucar.geometry.y
saih_jucar["nombre_norm"] = saih_jucar["DESCRIPCIO"].apply(normalizar)

print(f"Estaciones SAIH-Júcar (las 3 capas):")
print(saih_jucar["TIPO_CAPA"].value_counts())

Estaciones SAIH-Júcar (las 3 capas):
TIPO_CAPA
PLUVIOMETRO    124
EMBALSE         67
AFORO           38
Name: count, dtype: int64


In [13]:
print("=== AFOROS del Júcar (DESCRIPCIO) ===")
aforos = saih_jucar[saih_jucar["TIPO_CAPA"] == "AFORO"]["DESCRIPCIO"].tolist()
for a in aforos:
    print(f"  {a}")

=== AFOROS del Júcar (DESCRIPCIO) ===
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan
  nan


In [15]:
print("Columnas pluviometria:", list(pluv.columns))
print("Columnas embalses:    ", list(emba.columns))
print("Columnas caudalrios:  ", list(caud.columns))

Columnas pluviometria: ['OBJECTID', 'IDPUNTO', 'DESCRIPCIO', 'IDSAIH', 'TIPO', 'geometry', 'TIPO_CAPA']
Columnas embalses:     ['OBJECTID', 'IDPUNTO', 'IDSAIH', 'TIPO', 'geometry', 'TIPO_CAPA']
Columnas caudalrios:   ['OBJECTID', 'IDPUNTO', 'IDSAIH', 'TIPO', 'geometry', 'TIPO_CAPA']


In [16]:
emba.head()

,OBJECTID,IDPUNTO,IDSAIH,TIPO,geometry,TIPO_CAPA
0,229,E_40 JERTE-PLASENCIA,TAJO,Capacidad embalse (%),POINT (-6.04381 40.06231),EMBALSE
1,234,E031Z65VEMBA,EBRO,Capacidad embalse (%),POINT (-1.15861 42.98607),EMBALSE
2,237,E020C65VEMBA,EBRO,Capacidad embalse (%),POINT (-0.41476 40.87583),EMBALSE
3,240,E_47 CEDILLO,TAJO,Capacidad embalse (%),POINT (-7.53989 39.66439),EMBALSE
4,246,E65_GERGAL,GUADALQUIVIR,Capacidad embalse (%),POINT (-6.0512 37.56746),EMBALSE


In [17]:
caud.head()

,OBJECTID,IDPUNTO,IDSAIH,TIPO,geometry,TIPO_CAPA
0,127,048C02CC,ACA,Caudal río (m3/s),POINT (2.95447 42.15454),AFORO
1,128,048R01CR,ACA,Caudal río (m3/s),POINT (2.95447 42.15454),AFORO
2,130,083R01CR,ACA,Caudal río (m3/s),POINT (2.81901 41.99119),AFORO
3,133,060C02CC,ACA,Caudal río (m3/s),POINT (2.19488 42.17328),AFORO
4,134,060R01CR,ACA,Caudal río (m3/s),POINT (2.19488 42.17328),AFORO


In [14]:
# ¿Está Chiva/Forata/Poyo/Turís en los matches?
estaciones_criticas = ["CHIVA", "FORATA", "POYO", "TURIS"]
for est in estaciones_criticas:
    en_shp = saih_jucar[saih_jucar["nombre_norm"].str.contains(est, na=False)]
    print(f"\n=== {est} en el shapefile ===")
    print(en_shp[["DESCRIPCIO", "TIPO_CAPA", "lat", "lon"]].to_string(index=False))


=== CHIVA en el shapefile ===
Empty DataFrame
Columns: [DESCRIPCIO, TIPO_CAPA, lat, lon]
Index: []

=== FORATA en el shapefile ===
DESCRIPCIO   TIPO_CAPA       lat       lon
    FORATA PLUVIOMETRO 39.341183 -0.864723

=== POYO en el shapefile ===
 DESCRIPCIO   TIPO_CAPA       lat     lon
RAMBLA POYO PLUVIOMETRO 39.475371 -0.5818

=== TURIS en el shapefile ===
Empty DataFrame
Columns: [DESCRIPCIO, TIPO_CAPA, lat, lon]
Index: []


In [18]:
# ¿Las estaciones clave de la DANA están con coordenadas?
saih_pdf = pd.read_parquet("../data/processed/saih_historico_2020_2024.parquet")

claves = ["CHIVA", "FORATA", "POYO", "TURIS", "UTIEL", "TOUS", "ANTELLA",
          "BUSEO", "LORIGUILLA", "ALARCON", "ULLDECONA"]

descripciones = (saih_pdf["descripcion"].drop_duplicates().to_frame())
descripciones["match"] = descripciones["descripcion"].apply(buscar_match)

print("=== Estaciones clave del evento DANA ===")
for clave in claves:
    rows = descripciones[descripciones["descripcion"].str.contains(clave, case=False, na=False)]
    for _, row in rows.iterrows():
        status = "✓" if pd.notna(row["match"]) else "✗ SIN MATCH"
        print(f"  {status}  {row['descripcion']:50s} → {row['match']}")

=== Estaciones clave del evento DANA ===
  ✗ SIN MATCH  Pluviometro de Chiva Chiva                         → nan
  ✓  Embalse de Forata Yátova                           → FORATA
  ✗ SIN MATCH  Marco en rambla del Poyo Riba-roja                 → nan
  ✓  Pluvionivometro Remedio de Utiel Utiel             → UTIEL
  ✓  Embalse de Tous Tous                               → TOUS
  ✓  Azud de Antella Antella                            → ANTELLA
  ✓  Embalse de Buseo Chera                             → BUSEO
  ✓  Embalse de Loriguilla Loriguilla                   → LORIGUILLA
  ✓  Embalse de Ulldecona La Pobla de Benifassa         → ULLDECONA
  ✓  Embalse de Ulldecona La Pobla Benifassa(ULL)       → ULLDECONA


In [19]:
# Diccionario de overrides manuales para estaciones críticas
# que no se matchean automáticamente por discrepancias en la cartografía oficial.
OVERRIDES = {
    "Pluviometro de Chiva Chiva": "CHIVA",
    "Marco en rambla del Poyo Riba-roja": "POYO",
}

def buscar_match_completo(descripcion_pdf):
    # 1) Overrides primero
    for clave, nombre_norm in OVERRIDES.items():
        if clave in descripcion_pdf:
            return nombre_norm
    # 2) Luego matching automático
    return buscar_match(descripcion_pdf)

# Verifica que ahora sí las críticas están
descripciones["match_final"] = descripciones["descripcion"].apply(buscar_match_completo)

print("=== Críticas tras override manual ===")
for clave in ["CHIVA", "POYO"]:
    rows = descripciones[descripciones["descripcion"].str.contains(clave, case=False, na=False)]
    for _, row in rows.iterrows():
        print(f"  ✓  {row['descripcion']:55s} → {row['match_final']}")

=== Críticas tras override manual ===
  ✓  Pluviometro de Chiva Chiva                              → CHIVA
  ✓  Marco en rambla del Poyo Riba-roja                      → POYO


In [35]:
saih_jucar.head()
#print(saih_jucar["nombre_norm"].unique())

,OBJECTID,IDPUNTO,DESCRIPCIO,IDSAIH,TIPO,geometry,TIPO_CAPA,lon,lat,nombre_norm
53,872,PLUVIO VALLIBONA,VALLIBONA,JUCAR,Pluviometría (mm),POINT (0.00111 40.58479),PLUVIOMETRO,0.001111,40.584792,VALLIBONA
54,874,PLUVIO VALBONA,VALBONA,JUCAR,Pluviometría (mm),POINT (-0.81806 40.25407),PLUVIOMETRO,-0.818062,40.254068,VALBONA
55,875,PLUVIO PINA,PINA,JUCAR,Pluviometría (mm),POINT (-0.62812 40.02923),PLUVIOMETRO,-0.628123,40.029234,PINA
56,876,PLUVIO ALBARRACIN,ALBARRACIN,JUCAR,Pluviometría (mm),POINT (-1.43605 40.43102),PLUVIOMETRO,-1.436050,40.431020,ALBARRACIN
57,880,PLUVIO LANDETE,LANDETE,JUCAR,Pluviometría (mm),POINT (-1.37154 39.91162),PLUVIOMETRO,-1.371545,39.911618,LANDETE


In [22]:
for nombre in ["CHIVA", "POYO"]:
    rows = saih_jucar[saih_jucar["nombre_norm"].str.contains(nombre, na=False)]
    print(f"\n=== {nombre} en shapefile ===")
    print(rows[["nombre_norm", "TIPO_CAPA", "lat", "lon"]].to_string(index=False))


=== CHIVA en shapefile ===
Empty DataFrame
Columns: [nombre_norm, TIPO_CAPA, lat, lon]
Index: []

=== POYO en shapefile ===
nombre_norm   TIPO_CAPA       lat     lon
RAMBLA POYO PLUVIOMETRO 39.475371 -0.5818


In [36]:
saih_all.head()

,OBJECTID,IDPUNTO,DESCRIPCIO,IDSAIH,TIPO,geometry,TIPO_CAPA
0,963,I07_SALADO_MENG,I07_SALADO_MENG,GUADALQUIVIR,Pluviometría (mm),POINT (-3.8075 37.98144),PLUVIOMETRO
1,978,AR49 AZUD DEL ENCIN,AR49 AZUD DEL ENCIN,TAJO,Pluviometría (mm),POINT (-6.42594 39.94885),PLUVIOMETRO
2,994,AR22 COFIO EN SAN MARTIN,AR22 COFIO EN SAN MARTIN,TAJO,Pluviometría (mm),POINT (-4.32259 40.41204),PLUVIOMETRO
3,1011,E_11 BELENA,E_11 BELENA,TAJO,Pluviometría (mm),POINT (-3.19433 40.93034),PLUVIOMETRO
4,1033,EM3-02,CUMBRE MAYORES,GUADIANA,Pluviometría (mm),POINT (-6.64619 38.05858),PLUVIOMETRO


In [37]:
# Buscar Chiva en TODO el shapefile (no solo Júcar, por si está en otra demarcación)
buscar = saih_all[
    saih_all["DESCRIPCIO"].astype(str).str.contains("CHIVA", case=False, na=False)
]

print(f"Filas con 'CHIVA' en el shapefile entero: {len(buscar)}")
if len(buscar) > 0:
    print(buscar[["DESCRIPCIO", "IDPUNTO", "IDSAIH", "TIPO_CAPA", "geometry"]].to_string(index=False))
else:
    print("CHIVA NO está en el shapefile.")

Filas con 'CHIVA' en el shapefile entero: 0
CHIVA NO está en el shapefile.


In [38]:
for nombre_buscar in ["POYO", "TURIS", "TURÍS"]:
    encontrado = saih_all[
        saih_all["DESCRIPCIO"].astype(str).str.contains(nombre_buscar, case=False, na=False)
        | saih_all["IDPUNTO"].astype(str).str.contains(nombre_buscar, case=False, na=False)
    ]
    print(f"\n=== {nombre_buscar} en todo el shapefile ===")
    if len(encontrado) > 0:
        print(encontrado[["DESCRIPCIO", "IDPUNTO", "IDSAIH", "TIPO_CAPA"]].to_string(index=False))
    else:
        print(f"  {nombre_buscar} NO está.")


=== POYO en todo el shapefile ===
 DESCRIPCIO             IDPUNTO IDSAIH   TIPO_CAPA
RAMBLA POYO PLUVIO  RAMBLA POYO  JUCAR PLUVIOMETRO
        NaN  AFORO  RAMBLA POYO  JUCAR       AFORO

=== TURIS en todo el shapefile ===
  TURIS NO está.

=== TURÍS en todo el shapefile ===
  TURÍS NO está.


In [41]:
OVERRIDES = {
    "Pluviometro de Chiva Chiva":          "CHIVA",
    "Marco en rambla del Poyo Riba-roja":  "RAMBLA POYO",
    "MC TURIS":                            "TURIS",
}

In [40]:
import requests
import time

def geocodificar_municipio(nombre_municipio: str, provincia: str = "España"):
    """Devuelve (lat, lon) del centro del municipio usando Nominatim OSM."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": f"{nombre_municipio}, {provincia}",
        "format": "json",
        "limit": 1,
    }
    headers = {"User-Agent": "tfm-flood-warning/1.0"}  # Nominatim lo exige
    r = requests.get(url, params=params, headers=headers, timeout=15)
    r.raise_for_status()
    data = r.json()
    if not data:
        return None, None
    return float(data[0]["lat"]), float(data[0]["lon"])


# Pedimos las coordenadas de los municipios que faltan
faltantes = {
    "CHIVA": ("Chiva", "Valencia, España"),
    "TURIS": ("Turís", "Valencia, España"),
}

COORDENADAS_MANUALES = {}
for clave, (municipio, provincia) in faltantes.items():
    lat, lon = geocodificar_municipio(municipio, provincia)
    COORDENADAS_MANUALES[clave] = {
        "lat": lat,
        "lon": lon,
        "fuente": "OpenStreetMap/Nominatim (municipio)",
    }
    print(f"  {clave} → {municipio}: ({lat:.4f}, {lon:.4f})")
    time.sleep(1)  # Nominatim limita a 1 petición/segundo

  CHIVA → Chiva: (39.4726, -0.7178)
  TURIS → Turís: (39.3899, -0.7105)


"Para las estaciones críticas Chiva y Turís, ausentes en la cartografía oficial del MITECO, las coordenadas se obtuvieron mediante geocodificación del municipio correspondiente a través del servicio público Nominatim de OpenStreetMap. La precisión resultante (~1 km, equivalente al centro geográfico del municipio) se considera aceptable para visualizaciones a escala provincial. Fuente: © OpenStreetMap contributors, licencia ODbL."

In [42]:
import pandas as pd

# Diccionario rápido de coordenadas del shapefile (Júcar)
coords_shp = dict(zip(saih_jucar["nombre_norm"], 
                      zip(saih_jucar["lat"], saih_jucar["lon"])))


def obtener_coords(nombre_match):
    """Devuelve (lat, lon, fuente) a partir del nombre matcheado."""
    if pd.isna(nombre_match):
        return None, None, "sin_match"
    if nombre_match in COORDENADAS_MANUALES:
        m = COORDENADAS_MANUALES[nombre_match]
        return m["lat"], m["lon"], m["fuente"]
    if nombre_match in coords_shp:
        lat, lon = coords_shp[nombre_match]
        return lat, lon, "shapefile_miteco"
    return None, None, "match_sin_coords"


# Aplicar matching a todas las descripciones del PDF
descripciones = saih_pdf["descripcion"].drop_duplicates().to_frame().reset_index(drop=True)
descripciones["match_final"] = descripciones["descripcion"].apply(buscar_match_completo)
descripciones[["lat", "lon", "fuente_coords"]] = descripciones["match_final"].apply(
    lambda m: pd.Series(obtener_coords(m))
)

# Resumen
total = len(descripciones)
con_coords = descripciones["lat"].notna().sum()
print(f"Descripciones únicas: {total}")
print(f"  Con coordenadas:  {con_coords} ({con_coords/total*100:.1f}%)")
print(f"  Sin coordenadas:  {total - con_coords}")
print()
print("Origen de las coordenadas:")
print(descripciones["fuente_coords"].value_counts())

Descripciones únicas: 186
  Con coordenadas:  115 (61.8%)
  Sin coordenadas:  71

Origen de las coordenadas:
fuente_coords
shapefile_miteco                       114
sin_match                               71
OpenStreetMap/Nominatim (municipio)      1
Name: count, dtype: int64


In [43]:
print("=== Estaciones clave del evento DANA ===")
for clave in ["CHIVA", "POYO", "TURIS", "FORATA", "UTIEL", "TOUS", "ANTELLA"]:
    rows = descripciones[descripciones["descripcion"].str.contains(clave, case=False, na=False)]
    for _, row in rows.iterrows():
        status = "✓" if pd.notna(row["lat"]) else "✗"
        coords = f"({row['lat']:.3f}, {row['lon']:.3f})" if pd.notna(row["lat"]) else "SIN COORDS"
        print(f"  {status} {row['descripcion']:55s} → {coords}")

=== Estaciones clave del evento DANA ===
  ✓ Pluviometro de Chiva Chiva                              → (39.473, -0.718)
  ✓ Marco en rambla del Poyo Riba-roja                      → (39.475, -0.582)
  ✓ Embalse de Forata Yátova                                → (39.341, -0.865)
  ✓ Pluvionivometro Remedio de Utiel Utiel                  → (39.589, -1.126)
  ✓ Embalse de Tous Tous                                    → (39.130, -0.648)
  ✓ Azud de Antella Antella                                 → (39.081, -0.597)


In [46]:
# Unir el dataset histórico con las coordenadas obtenidas
mapping = descripciones.set_index("descripcion")[["match_final", "lat", "lon", "fuente_coords"]]

saih_geo = saih_pdf.merge(
    mapping,
    left_on="descripcion",
    right_index=True,
    how="left",
)

# Guardar como nuevo Parquet
out_path = Path("../data/processed/saih_geolocalizado_2020_2024.parquet")
saih_geo.to_parquet(out_path, index=False)

print(f"Guardado: {out_path}")
print(f"  Filas totales: {len(saih_geo):,}")
print(f"  Filas con coordenadas: {saih_geo['lat'].notna().sum():,}")
print(f"  Tamaño: {out_path.stat().st_size / 1024:.1f} KB")

Guardado: ..\data\processed\saih_geolocalizado_2020_2024.parquet
  Filas totales: 31,697
  Filas con coordenadas: 19,747
  Tamaño: 89.6 KB


In [47]:
# % real de filas con coordenadas en el dataset histórico completo
total_filas = len(saih_geo)
con_coords_filas = saih_geo["lat"].notna().sum()
sin_coords_filas = total_filas - con_coords_filas

print(f"Filas totales: {total_filas:,}")
print(f"  Con coordenadas: {con_coords_filas:,} ({con_coords_filas/total_filas*100:.1f}%)")
print(f"  Sin coordenadas: {sin_coords_filas:,} ({sin_coords_filas/total_filas*100:.1f}%)")
print()
print("=== Top 10 estaciones sin coordenadas (por nº de filas) ===")
print(saih_geo[saih_geo["lat"].isna()]["descripcion"].value_counts().head(10))

Filas totales: 31,697
  Con coordenadas: 19,747 (62.3%)
  Sin coordenadas: 11,950 (37.7%)

=== Top 10 estaciones sin coordenadas (por nº de filas) ===
descripcion
Aforo en Montaverner Montaverner                        180
Aforo en Los Frailes Valdeganga                         180
Marco en rio Cabriel Villatoya                          180
Pluviómetro de Cerro del Aguila Fuentealbilla           180
Pluviómetro de Font de la Figuera Font de la Figuera    180
Pluviometro de Millena Millena                          180
Pluviómetro de Sax Sax                                  180
Pluvionivometro Ciudad Encantada Valdecabras            179
Marco Villanueva Castellon Villanueva Castellon         179
Marco en Quintanar del Rey Quintanar del Rey            179
Name: count, dtype: int64
